In [ ]:
# Risk Monitor - Analyse exploratoire des données

In [ ]:
## Imports et connexion à la base

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

DB_PATH = Path("../data/risk_monitor_dataset.sqlite")
conn = sqlite3.connect(DB_PATH)

print("Base trouvée :", DB_PATH.exists())
print("Chemin :", DB_PATH)

Base trouvée : True
Chemin : ..\data\risk_monitor_dataset.sqlite


In [ ]:
## Lister les tables

In [2]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", conn)

tables

,name
0,complaints
1,memberships
2,payments
3,subscriptions
4,users


In [ ]:
## Compter le nombre de lignes par table

In [3]:
table_names = tables["name"].tolist()

row_counts = []

for table in table_names:
    count_df = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table}", conn)
    row_counts.append({
        "table": table,
        "row_count": int(count_df.loc[0, "row_count"])
    })

row_counts_df = pd.DataFrame(row_counts)
row_counts_df

,table,row_count
0,complaints,1213
1,memberships,1083
2,payments,7277
3,subscriptions,400
4,users,2001


In [ ]:
## Voir les colonnes de chaque table

In [4]:
for table in table_names:
    print(f"\n===== {table.upper()} =====")
    columns_df = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
    display(columns_df)


===== COMPLAINTS =====


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,reporter_id,INTEGER,0,None,0
2,2,target_id,INTEGER,0,None,0
3,3,subscription_id,INTEGER,0,None,0
4,4,type,TEXT,0,None,0
5,5,status,TEXT,0,None,0
6,6,created_at,TEXT,0,None,0
7,7,resolved_at,TEXT,0,None,0
8,8,resolution,TEXT,0,None,0



===== MEMBERSHIPS =====


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,0
1,1,user_id,INTEGER,0,None,0
2,2,subscription_id,INTEGER,0,None,0
3,3,status,INTEGER,0,None,0
4,4,joined_at,TEXT,0,None,0
5,5,left_at,TEXT,0,None,0
6,6,reason,TEXT,0,None,0



===== PAYMENTS =====


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,user_id,INTEGER,0,None,0
2,2,subscription_id,INTEGER,0,None,0
3,3,amount_cents,INTEGER,0,None,0
4,4,fee_cents,INTEGER,0,None,0
5,5,status,TEXT,0,None,0
6,6,created_at,TEXT,0,None,0
7,7,captured_at,TEXT,0,None,0
8,8,currency,TEXT,0,None,0
9,9,stripe_error_code,TEXT,0,None,0



===== SUBSCRIPTIONS =====


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,brand,TEXT,0,None,0
2,2,owner_id,INTEGER,0,None,0
3,3,created_at,TEXT,0,None,0
4,4,status,INTEGER,0,None,0
5,5,max_slots,INTEGER,0,None,0
6,6,price_cents,INTEGER,0,None,0
7,7,currency,TEXT,0,None,0



===== USERS =====


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,email,TEXT,0,None,0
2,2,country,TEXT,0,None,0
3,3,signup_date,TEXT,0,None,0
4,4,status,INTEGER,0,None,0
5,5,last_seen,TEXT,0,None,0
6,6,referral_code,TEXT,0,None,0
7,7,phone_prefix,TEXT,0,None,0


In [ ]:
## Voir 5 lignes d’exemple par table

In [5]:
for table in table_names:
    print(f"\n===== APERÇU : {table.upper()} =====")
    preview_df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5", conn)
    display(preview_df)


===== APERÇU : COMPLAINTS =====


,id,reporter_id,target_id,subscription_id,type,status,created_at,resolved_at,resolution
0,1,904,1565,389,billing_issue,open,2023-09-06 08:02:33,NaN,NaN
1,2,1702,600,260,subscription_inactive,escalated,2024-07-30 12:44:00,NaN,NaN
2,3,1361,672,124,Accès refusé,resolved,2022-10-01 21:19:13,2024-01-27 20:51:20,
3,4,19,1706,317,ACCESS_DENIED,RESOLVED,2024-05-08 04:05:41,2024-06-03 03:41:59,refunded
4,5,327,609,289,ACCESS_DENIED,closed,2021-06-11 13:01:44,2021-10-31 06:56:01,refunded



===== APERÇU : MEMBERSHIPS =====


,id,user_id,subscription_id,status,joined_at,left_at,reason
0,563,1485,231,1,2024-12-06 21:23:01,NaN,NaN
1,1042,690,399,2,2024-09-23 02:25:46,2025-03-31 22:56:44,fraud
2,72,56,31,1,2025-02-15 18:19:15,NaN,NaN
3,414,590,168,1,2024-08-06 15:33:32,NaN,NaN
4,413,20,168,1,2024-05-19 10:34:05,NaN,NaN



===== APERÇU : PAYMENTS =====


,id,user_id,subscription_id,amount_cents,fee_cents,status,created_at,captured_at,currency,stripe_error_code
0,1,1485,231,1322,78,succeeded,2024-12-04T21:23:01+02:00,2024-12-07 14:23:01,EUR,None
1,2,1485,231,366,37,succeeded,2025-01-03T21:23:01+01:00,2025-01-05 11:23:01,eur,None
2,3,1485,231,238,28,succeeded,2025-02-07 21:23:01,2025-02-10 04:23:01,EUR,None
3,4,1485,231,212,23,succeeded,2025-03-04 21:23:01,2025-03-06 20:23:01,EUR,None
4,5,1485,231,1325,144,succeeded,2025-04-03 21:23:01,2025-04-04 23:23:01,EUR,None



===== APERÇU : SUBSCRIPTIONS =====


,id,brand,owner_id,created_at,status,max_slots,price_cents,currency
0,1,Microsoft 365,1,2023-01-14 23:23:57,0,4,1070,eur
1,2,HBO Max,1670,2023-07-16 21:39:51,2,4,233,USD
2,3,ChatGPT Plus,558,2024-10-02 13:54:25,0,3,1367,EUR
3,4,Microsoft 365,567,2023-10-26 23:03:50,0,6,460,EUR
4,5,NordVPN,950,2024-11-30 08:35:40,0,4,464,EUR



===== APERÇU : USERS =====


,id,email,country,signup_date,status,last_seen,referral_code,phone_prefix
0,1,user_1@yahoo.fr,IT,2021-02-09 21:26:47,1,2021-04-28 14:23:00,NaN,+49
1,2,user_2@gmail.com,CH,2022-01-09T06:49:44Z,0,2023-01-05 17:03:20,NaN,+41
2,3,user_3@outlook.com,BE,1584765297,1,2021-07-30 05:38:37,27cb14dc,+32
3,4,user_4@hotmail.com,DE,2020-07-13T23:53:58Z,0,2024-06-10 09:06:04,NaN,+49
4,5,user_5@outlook.com,DE,21/07/2020 08:54,0,2026-03-06 20:16:50,NaN,+49


In [ ]:
## Compter les valeurs manquantes par table

In [6]:
for table in table_names:
    print(f"\n===== VALEURS MANQUANTES : {table.upper()} =====")
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    missing_df = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percent": (df.isna().sum().values / len(df) * 100).round(2)
    }).sort_values("missing_count", ascending=False)
    display(missing_df)


===== VALEURS MANQUANTES : COMPLAINTS =====


,column,missing_count,missing_percent
8,resolution,765,63.07
7,resolved_at,700,57.71
0,id,0,0.00
1,reporter_id,0,0.00
2,target_id,0,0.00
4,type,0,0.00
3,subscription_id,0,0.00
6,created_at,0,0.00
5,status,0,0.00



===== VALEURS MANQUANTES : MEMBERSHIPS =====


,column,missing_count,missing_percent
6,reason,706,65.19
5,left_at,637,58.82
0,id,0,0.00
2,subscription_id,0,0.00
1,user_id,0,0.00
4,joined_at,0,0.00
3,status,0,0.00



===== VALEURS MANQUANTES : PAYMENTS =====


,column,missing_count,missing_percent
9,stripe_error_code,5444,74.81
7,captured_at,3370,46.31
1,user_id,0,0.00
0,id,0,0.00
2,subscription_id,0,0.00
3,amount_cents,0,0.00
5,status,0,0.00
4,fee_cents,0,0.00
6,created_at,0,0.00
8,currency,0,0.00



===== VALEURS MANQUANTES : SUBSCRIPTIONS =====


,column,missing_count,missing_percent
0,id,0,0.0
1,brand,0,0.0
2,owner_id,0,0.0
3,created_at,0,0.0
4,status,0,0.0
5,max_slots,0,0.0
6,price_cents,0,0.0
7,currency,0,0.0



===== VALEURS MANQUANTES : USERS =====


,column,missing_count,missing_percent
6,referral_code,1418,70.86
7,phone_prefix,419,20.94
4,status,25,1.25
0,id,0,0.00
3,signup_date,0,0.00
2,country,0,0.00
1,email,0,0.00
5,last_seen,0,0.00


In [ ]:
## Regarder les valeurs distinctes de statut

In [7]:
status_columns = {
    "users": ["status"],
    "subscriptions": ["status"],
    "memberships": ["status", "reason"],
    "payments": ["status", "currency", "stripe_error_code"],
    "complaints": ["type", "status", "resolution"]
}

for table, columns in status_columns.items():
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    print(f"\n\n########## {table.upper()} ##########")
    for col in columns:
        print(f"\n--- {col} ---")
        value_counts = df[col].astype(str).value_counts(dropna=False).reset_index()
        value_counts.columns = [col, "count"]
        display(value_counts.head(20))



########## USERS ##########

--- status ---


,status,count
0,0.0,1325
1,4.0,198
2,1.0,179
3,3.0,110
4,2.0,107
5,-1.0,38
6,NaN,25
7,99.0,19




########## SUBSCRIPTIONS ##########

--- status ---


,status,count
0,0,229
1,2,68
2,3,57
3,1,46




########## MEMBERSHIPS ##########

--- status ---


...

,type,count
0,other,153
1,wrong_credentials,145
2,subscription_inactive,143
3,owner_unresponsive,142
4,ACCESS_DENIED,135
5,access_denied,134
6,billing_issue,127
7,fraud_suspicion,118
8,Accès refusé,116



--- status ---


,status,count
0,in_progress,194
1,open,192
2,RESOLVED,171
3,resolved,169
4,escalated,165
5,Open,164
6,closed,158



--- resolution ---


,resolution,count
0,NaN,765
1,,78
2,owner_warned,78
3,no_action,78
4,account_banned,74
5,refunded,71
6,subscriber_replaced,69


In [ ]:
## Regarder les colonnes de dates

In [8]:
date_columns = {
    "users": ["signup_date", "last_seen"],
    "subscriptions": ["created_at"],
    "memberships": ["joined_at", "left_at"],
    "payments": ["created_at", "captured_at"],
    "complaints": ["created_at", "resolved_at"]
}

for table, columns in date_columns.items():
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    print(f"\n\n########## {table.upper()} ##########")
    for col in columns:
        print(f"\n--- {col} : exemples ---")
        display(df[[col]].drop_duplicates().head(10))



########## USERS ##########

--- signup_date : exemples ---


,signup_date
0,2021-02-09 21:26:47
1,2022-01-09T06:49:44Z
2,1584765297
3,2020-07-13T23:53:58Z
4,21/07/2020 08:54
5,2023-03-09 18:54:13
6,2024-11-25T09:10:59Z
7,2022-11-03 16:05:01
8,2025-02-15T23:22:07Z
9,2025-03-26T08:39:40Z



--- last_seen : exemples ---


,last_seen
0,2021-04-28 14:23:00
1,2023-01-05 17:03:20
2,2021-07-30 05:38:37
3,2024-06-10 09:06:04
4,2026-03-06 20:16:50
5,2024-01-29 05:53:30
6,2025-05-14 23:06:23
7,2023-09-29 23:22:32
8,2025-12-23 04:44:01
9,2025-09-24 03:12:08




########## SUBSCRIPTIONS ##########

--- created_at : exemples ---


...

,captured_at
0,2024-12-07 14:23:01
1,2025-01-05 11:23:01
2,2025-02-10 04:23:01
3,2025-03-06 20:23:01
4,2025-04-04 23:23:01
5,2025-05-05 09:23:01
6,NaN
11,2025-02-21 03:25:46
12,2025-03-21 21:25:46
13,2025-04-23 11:25:46




########## COMPLAINTS ##########

--- created_at : exemples ---


,created_at
0,2023-09-06 08:02:33
1,2024-07-30 12:44:00
2,2022-10-01 21:19:13
3,2024-05-08 04:05:41
4,2021-06-11 13:01:44
5,2024-09-15 02:28:54
6,2023-02-09 22:32:02
7,2022-05-10 04:21:43
8,2023-07-07 13:17:22
9,2025-04-24 17:49:55



--- resolved_at : exemples ---


,resolved_at
0,NaN
2,2024-01-27 20:51:20
3,2024-06-03 03:41:59
4,2021-10-31 06:56:01
6,2024-10-07 04:55:46
8,2025-05-02 18:45:20
12,2024-05-28 02:59:01
14,2025-05-13 20:06:35
17,2025-03-03 23:13:08
18,2025-05-22 03:51:34


In [ ]:
## Rechercher les doublons exacts

In [9]:
for table in table_names:
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    duplicate_count = df.duplicated().sum()
    print(f"{table}: {duplicate_count} doublons exacts")

complaints: 0 doublons exacts
memberships: 0 doublons exacts
payments: 0 doublons exacts
subscriptions: 0 doublons exacts
users: 0 doublons exacts


In [1]:
## Premières observations

#La base contient 5 tables principales : `users`, `subscriptions`, `memberships`, `payments` et `complaints`.  
#Par exemple, `users` contient les informations générales sur les utilisateurs, `payments` contient les paiements, et `complaints` contient les réclamations.

#Plusieurs champs de statut semblent incohérents.  
#Par exemple, dans `complaints.status`, on observe des valeurs comme `open`, `closed`, `resolved`, `RESOLVED` et `escalated`. Cela montre qu’un même type d’information n’est pas toujours écrit de manière uniforme.

#Certaines colonnes de dates contiennent visiblement des formats mixtes.  
#Par exemple, on rencontre des valeurs comme `2021-02-09 21:26:47`, `2022-01-09T06:49:44Z`, `1584765297` ou encore `21/07/2020 08:54`. Ces différences devront être corrigées avant toute analyse temporelle fiable.

#Certaines colonnes catégorielles devront probablement être normalisées.  
#Par exemple, dans `complaints.type`, on trouve à la fois des valeurs comme `billing_issue`, `subscription_inactive`, `Accès refusé` et `ACCESS_DENIED`. Cela suggère que plusieurs écritures différentes peuvent désigner un même type de problème.

#Des valeurs manquantes sont présentes dans plusieurs tables.  
#Par exemple, `captured_at` dans `payments`, `resolved_at` dans `complaints` ou `left_at` dans `memberships` peuvent être vides. Il faudra distinguer les absences normales des absences problématiques.

#Enfin, plusieurs tables utilisent des codes de statut numériques non documentés, notamment `users`, `subscriptions` et `memberships`. Leur signification devra être déduite à partir du contexte et des autres colonnes disponibles.

In [7]:
## Analyse des valeurs manquantes

In [ ]:
# Valeurs manquantes par table

In [6]:
table_names = ["users", "subscriptions", "memberships", "payments", "complaints"]

for table in table_names:
    print(f"\n===== {table.upper()} =====")
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)

    missing_df = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percent": ((df.isna().sum().values / len(df)) * 100).round(2)
    }).sort_values(by="missing_count", ascending=False)

    display(missing_df)


===== USERS =====


,column,missing_count,missing_percent
6,referral_code,1418,70.86
7,phone_prefix,419,20.94
4,status,25,1.25
0,id,0,0.00
3,signup_date,0,0.00
2,country,0,0.00
1,email,0,0.00
5,last_seen,0,0.00



===== SUBSCRIPTIONS =====


,column,missing_count,missing_percent
0,id,0,0.0
1,brand,0,0.0
2,owner_id,0,0.0
3,created_at,0,0.0
4,status,0,0.0
5,max_slots,0,0.0
6,price_cents,0,0.0
7,currency,0,0.0



===== MEMBERSHIPS =====


,column,missing_count,missing_percent
6,reason,706,65.19
5,left_at,637,58.82
0,id,0,0.00
2,subscription_id,0,0.00
1,user_id,0,0.00
4,joined_at,0,0.00
3,status,0,0.00



===== PAYMENTS =====


,column,missing_count,missing_percent
9,stripe_error_code,5444,74.81
7,captured_at,3370,46.31
1,user_id,0,0.00
0,id,0,0.00
2,subscription_id,0,0.00
3,amount_cents,0,0.00
5,status,0,0.00
4,fee_cents,0,0.00
6,created_at,0,0.00
8,currency,0,0.00



===== COMPLAINTS =====


,column,missing_count,missing_percent
8,resolution,765,63.07
7,resolved_at,700,57.71
0,id,0,0.00
1,reporter_id,0,0.00
2,target_id,0,0.00
4,type,0,0.00
3,subscription_id,0,0.00
6,created_at,0,0.00
5,status,0,0.00


In [9]:
# Premières interprétation des valeurs manquantes

In [10]:
# Toutes les valeurs manquantes ne sont pas forcément des erreurs.

# Quelques absences peuvent être normales :
# - `left_at` peut être vide si le subscriber est encore actif
# - `resolved_at` peut être vide si la plainte est encore ouverte
# - `captured_at` peut être vide si le paiement a échoué

# En revanche, certaines absences peuvent être problématiques selon le contexte et devront être vérifiées plus tard.

In [12]:
# Valeurs distinctes pour les statuts et catégories

In [13]:
columns_to_check = {
    "users": ["status", "country", "phone_prefix"],
    "subscriptions": ["brand", "status", "currency"],
    "memberships": ["status", "reason"],
    "payments": ["status", "currency", "stripe_error_code"],
    "complaints": ["type", "status", "resolution"]
}

for table, columns in columns_to_check.items():
    print(f"\n\n########## {table.upper()} ##########")
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)

    for col in columns:
        print(f"\n--- {col} ---")
        counts = (
            df[col]
            .astype(str)
            .value_counts(dropna=False)
            .reset_index()
        )
        counts.columns = [col, "count"]
        display(counts.head(30))



########## USERS ##########

--- status ---


,status,count
0,0.0,1325
1,4.0,198
2,1.0,179
3,3.0,110
4,2.0,107
5,-1.0,38
6,NaN,25
7,99.0,19



--- country ---


,country,count
0,ES,165
1,AT,153
2,DE,149
3,IT,140
4,BE,135
5,France,134
6,fr,134
7,PT,133
8,,132
9,Fra,127



--- phone_prefix ---


...

,type,count
0,other,153
1,wrong_credentials,145
2,subscription_inactive,143
3,owner_unresponsive,142
4,ACCESS_DENIED,135
5,access_denied,134
6,billing_issue,127
7,fraud_suspicion,118
8,Accès refusé,116



--- status ---


,status,count
0,in_progress,194
1,open,192
2,RESOLVED,171
3,resolved,169
4,escalated,165
5,Open,164
6,closed,158



--- resolution ---


,resolution,count
0,NaN,765
1,,78
2,owner_warned,78
3,no_action,78
4,account_banned,74
5,refunded,71
6,subscriber_replaced,69


In [23]:
## Premières observations sur les statuts et les catégories

# Plusieurs colonnes catégorielles semblent nécessiter une harmonisation.

# Points à vérifier :
# - présence de majuscules et minuscules différentes pour une même idée
# - présence possible de variantes orthographiques
# - présence de valeurs en anglais et en français
# - présence de codes numériques non documentés

In [16]:
# Colonnes de dates, exemples uniques

In [17]:
date_columns = {
    "users": ["signup_date", "last_seen"],
    "subscriptions": ["created_at"],
    "memberships": ["joined_at", "left_at"],
    "payments": ["created_at", "captured_at"],
    "complaints": ["created_at", "resolved_at"]
}

for table, columns in date_columns.items():
    print(f"\n\n########## {table.upper()} ##########")
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)

    for col in columns:
        print(f"\n--- {col} : exemples ---")
        display(df[[col]].drop_duplicates().head(15))



########## USERS ##########

--- signup_date : exemples ---


,signup_date
0,2021-02-09 21:26:47
1,2022-01-09T06:49:44Z
2,1584765297
3,2020-07-13T23:53:58Z
4,21/07/2020 08:54
5,2023-03-09 18:54:13
6,2024-11-25T09:10:59Z
7,2022-11-03 16:05:01
8,2025-02-15T23:22:07Z
9,2025-03-26T08:39:40Z



--- last_seen : exemples ---


,last_seen
0,2021-04-28 14:23:00
1,2023-01-05 17:03:20
2,2021-07-30 05:38:37
3,2024-06-10 09:06:04
4,2026-03-06 20:16:50
5,2024-01-29 05:53:30
6,2025-05-14 23:06:23
7,2023-09-29 23:22:32
8,2025-12-23 04:44:01
9,2025-09-24 03:12:08




########## SUBSCRIPTIONS ##########

--- created_at : exemples ---


...

,captured_at
0,2024-12-07 14:23:01
1,2025-01-05 11:23:01
2,2025-02-10 04:23:01
3,2025-03-06 20:23:01
4,2025-04-04 23:23:01
5,2025-05-05 09:23:01
6,NaN
11,2025-02-21 03:25:46
12,2025-03-21 21:25:46
13,2025-04-23 11:25:46




########## COMPLAINTS ##########

--- created_at : exemples ---


,created_at
0,2023-09-06 08:02:33
1,2024-07-30 12:44:00
2,2022-10-01 21:19:13
3,2024-05-08 04:05:41
4,2021-06-11 13:01:44
5,2024-09-15 02:28:54
6,2023-02-09 22:32:02
7,2022-05-10 04:21:43
8,2023-07-07 13:17:22
9,2025-04-24 17:49:55



--- resolved_at : exemples ---


,resolved_at
0,NaN
2,2024-01-27 20:51:20
3,2024-06-03 03:41:59
4,2021-10-31 06:56:01
6,2024-10-07 04:55:46
8,2025-05-02 18:45:20
12,2024-05-28 02:59:01
14,2025-05-13 20:06:35
17,2025-03-03 23:13:08
18,2025-05-22 03:51:34


In [18]:
# Observations sur les dates

In [22]:
## Premières observations sur les colonnes de dates

# Les colonnes de dates semblent contenir plusieurs formats différents.

# Exemples de formats attendus dans la base :
# - format datetime classique
# - format ISO avec `T` et `Z`
# - timestamp numérique
# - format jour/mois/année

# Avant toute analyse temporelle fiable, ces colonnes devront être converties dans un format unique.

In [20]:
# Doublons exacts

In [21]:
duplicate_summary = []

for table in table_names:
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    duplicate_summary.append({
        "table": table,
        "exact_duplicates": int(df.duplicated().sum())
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)
duplicate_summary_df

,table,exact_duplicates
0,users,0
1,subscriptions,0
2,memberships,0
3,payments,0
4,complaints,0


In [3]:
conn.close()
print("Connexion fermée.")

Connexion fermée.


In [24]:
## Des anomalies prioritaires ont été identifiées.

#À ce stade, les anomalies suivantes semblent prioritaires :

# - formats de dates mixtes
# - statuts potentiellement incohérents
# - catégories écrites sous plusieurs formes
# - valeurs manquantes à interpréter selon le contexte
# - codes numériques non documentés dans certaines colonnes `status`

# La prochaine étape consistera à transformer ces constats en règles de nettoyage simples et explicites.